# Driver Drowsiness Detection using CNN + Attention

**Pipeline:**  
Image Input → Face Detection (MTCNN) → Face Crop + Resize → CNN Backbone (ResNet-50) → Attention Layer → FC → Sigmoid

**Dataset:** Driver Drowsiness Dataset (DDD) on Google Drive  
**Classes:** Drowsy / Non Drowsy

## Step 1 — Install Dependencies & Mount Drive

In [2]:
%pip install -q facenet-pytorch

from google.colab import drive
drive.mount('/content/drive')

import torch
print(f'PyTorch: {torch.__version__} | CUDA available: {torch.cuda.is_available()}')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 66.4 MB/s eta 0:00:00
PyTorch: 2.10.0+cu128 | CUDA available: True


## Step 2 — Configuration

In [3]:
import os, re, random, pathlib, pickle
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models
from PIL import Image
from facenet_pytorch import MTCNN
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score, roc_curve,
)
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DATA_ROOT = '/content/drive/MyDrive/datasets/Driver Drowsiness Dataset (DDD)'
PROCESSED_ROOT = '/content/drive/MyDrive/datasets/processed_faces'
ARTIFACTS_DIR = '/content/drive/MyDrive/datasets'
RESULTS_DIR = '/content/drive/MyDrive/datasets/pipeline_results'
os.makedirs(PROCESSED_ROOT, exist_ok=True)
os.makedirs(ARTIFACTS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

IMG_SIZE = 224
BATCH_SIZE = 32
NUM_EPOCHS = 15
LR = 1e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 2
VAL_RATIO = 0.2

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')
print(f'DATA_ROOT: {DATA_ROOT}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

Using device: cuda
GPU: Tesla T4


## Step 3 — Scan Dataset & Build Index
*[RUN ONCE — results saved to Drive]*

In [4]:
RECORDS_CACHE = os.path.join(RESULTS_DIR, 'step3_records.pkl')

if os.path.exists(RECORDS_CACHE):
    with open(RECORDS_CACHE, 'rb') as f:
        records = pickle.load(f)
    labels_arr = [r[1] for r in records]
    print(f'[CACHED] Loaded {len(records)} records from: {RECORDS_CACHE}')
    print(f'  Drowsy: {labels_arr.count(1)}  |  Non-Drowsy: {labels_arr.count(0)}')
    unique_seqs = sorted(set(r[2] for r in records))
    print(f'  Unique sequence prefixes: {len(unique_seqs)}')
else:
    CLASS_DIR_CANDIDATES = {
        1: ['Drowsy'],
        0: ['Non Drowsy', 'Non-Drowsy'],
    }
    PREFIX_RE = re.compile(r'^([A-Za-z]+)', re.IGNORECASE)

    DATA_ROOT = os.path.normpath(DATA_ROOT)
    print(f'DATA_ROOT: {DATA_ROOT}')
    if not os.path.isdir(DATA_ROOT):
        raise FileNotFoundError(f'DATA_ROOT does not exist: {DATA_ROOT}')

    resolved_class_dirs = {}
    for label, candidates in CLASS_DIR_CANDIDATES.items():
        found = None
        for name in candidates:
            p = os.path.join(DATA_ROOT, name)
            if os.path.isdir(p):
                found = (name, p)
                break
        if found is None:
            print(f'WARNING: no class directory found for label={label}; checked {candidates}')
        else:
            resolved_class_dirs[label] = found

    records = []
    for label, (cls_name, cls_dir) in resolved_class_dirs.items():
        print(f'Scanning: {cls_dir}')
        for fname in sorted(os.listdir(cls_dir)):
            if not fname.lower().endswith('.png'):
                continue
            m = PREFIX_RE.match(fname)
            seq_id = f'{cls_name}_{m.group(1).upper()}' if m else f'{cls_name}_UNKNOWN'
            records.append((os.path.join(cls_dir, fname), label, seq_id))

    labels_arr = [r[1] for r in records]
    print(f'Total images: {len(records)}')
    print(f'  Drowsy: {labels_arr.count(1)}  |  Non-Drowsy: {labels_arr.count(0)}')
    unique_seqs = sorted(set(r[2] for r in records))
    print(f'  Unique sequence prefixes: {len(unique_seqs)}')
    for s in unique_seqs[:10]:
        print(f'  {s}: {sum(1 for r in records if r[2] == s)} images')

    with open(RECORDS_CACHE, 'wb') as f:
        pickle.dump(records, f)
    print(f'Saved to: {RECORDS_CACHE}')

DATA_ROOT in kernel: D:\kagglehub_cache\datasets\ismailnasri20\driver-drowsiness-dataset-ddd\versions\1\DDD


FileNotFoundError: DATA_ROOT does not exist for this kernel: D:\kagglehub_cache\datasets\ismailnasri20\driver-drowsiness-dataset-ddd\versions\1\DDD

## Step 4 — Leakage-Safe Train / Val Split
*[RUN ONCE — results saved to Drive]*

Split by **sequence prefix** so frames from the same subject/clip never appear in both sets.

In [ ]:
SPLIT_CACHE = os.path.join(RESULTS_DIR, 'step4_split.pkl')

if os.path.exists(SPLIT_CACHE):
    with open(SPLIT_CACHE, 'rb') as f:
        split_data = pickle.load(f)
    train_records = split_data['train_records']
    val_records = split_data['val_records']
    print(f'[CACHED] Loaded split from: {SPLIT_CACHE}')
    print(f'Train: {len(train_records)} images  |  Val: {len(val_records)} images')
    print(f'Train drowsy ratio: {sum(r[1] for r in train_records)/len(train_records):.2%}')
    print(f'Val   drowsy ratio: {sum(r[1] for r in val_records)/len(val_records):.2%}')
else:
    seq_to_label = {}
    for _, label, seq_id in records:
        seq_to_label[seq_id] = label

    seq_ids = list(seq_to_label.keys())
    seq_labels = [seq_to_label[s] for s in seq_ids]

    train_seqs, val_seqs = train_test_split(
        seq_ids, test_size=VAL_RATIO, random_state=SEED, stratify=seq_labels
    )
    train_seqs_set = set(train_seqs)
    val_seqs_set = set(val_seqs)

    train_records = [r for r in records if r[2] in train_seqs_set]
    val_records   = [r for r in records if r[2] in val_seqs_set]

    print(f'Train: {len(train_records)} images  ({len(train_seqs)} sequences)')
    print(f'Val:   {len(val_records)} images  ({len(val_seqs)} sequences)')
    print(f'Train drowsy ratio: {sum(r[1] for r in train_records)/len(train_records):.2%}')
    print(f'Val   drowsy ratio: {sum(r[1] for r in val_records)/len(val_records):.2%}')

    with open(SPLIT_CACHE, 'wb') as f:
        pickle.dump({'train_records': train_records, 'val_records': val_records}, f)
    print(f'Saved to: {SPLIT_CACHE}')

## Step 5 — Face Detection (MTCNN) & Crop Cache
*[RUN ONCE — results saved to Drive]*

Run MTCNN **once** and save 224x224 face crops to Drive.  
Subsequent runs load cached path lists directly (no MTCNN needed).

In [ ]:
FACES_CACHE = os.path.join(RESULTS_DIR, 'step5_faces.pkl')

if os.path.exists(FACES_CACHE):
    with open(FACES_CACHE, 'rb') as f:
        faces_data = pickle.load(f)
    train_cached = faces_data['train_cached']
    val_cached = faces_data['val_cached']
    print(f'[CACHED] Loaded face path lists from: {FACES_CACHE}')
    print(f'  Train faces: {len(train_cached)}  |  Val faces: {len(val_cached)}')
else:
    mtcnn = MTCNN(
        image_size=IMG_SIZE,
        margin=40,
        keep_all=False,
        post_process=False,
        device=DEVICE,
    )

    os.makedirs(PROCESSED_ROOT, exist_ok=True)

    center_crop_fallback = T.Compose([
        T.Resize(IMG_SIZE + 32),
        T.CenterCrop(IMG_SIZE),
    ])

    def process_and_cache(record_list, desc='Processing'):
        cached_paths = []
        stats = {'mtcnn': 0, 'fallback': 0, 'cached': 0}
        for path, label, seq_id in tqdm(record_list, desc=desc):
            rel = os.path.relpath(path, DATA_ROOT)
            out_path = os.path.join(PROCESSED_ROOT, rel)
            os.makedirs(os.path.dirname(out_path), exist_ok=True)

            if os.path.exists(out_path):
                stats['cached'] += 1
                cached_paths.append((out_path, label))
                continue

            img = Image.open(path).convert('RGB')
            face = mtcnn(img)

            if face is not None:
                face_pil = T.ToPILImage()(face.byte())
                stats['mtcnn'] += 1
            else:
                face_pil = center_crop_fallback(img)
                stats['fallback'] += 1

            face_pil.save(out_path)
            cached_paths.append((out_path, label))

        print(f"  MTCNN detections: {stats['mtcnn']}  |  Fallback crops: {stats['fallback']}  |  Already cached: {stats['cached']}")
        return cached_paths

    print('--- Processing TRAIN set ---')
    train_cached = process_and_cache(train_records, desc='Train faces')
    print('--- Processing VAL set ---')
    val_cached = process_and_cache(val_records, desc='Val faces')

    with open(FACES_CACHE, 'wb') as f:
        pickle.dump({'train_cached': train_cached, 'val_cached': val_cached}, f)
    print(f'Saved to: {FACES_CACHE}')

## Step 6 — Dataset & DataLoaders

In [ ]:
train_transform = T.Compose([
    T.RandomHorizontalFlip(),
    T.RandomRotation(10),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]),
])

val_transform = T.Compose([
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]),
])


class DrowsinessDataset(Dataset):
    def __init__(self, cached_paths, transform=None):
        self.data = cached_paths
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        path, label = self.data[idx]
        img = Image.open(path).convert('RGB')
        img = img.resize((IMG_SIZE, IMG_SIZE))
        if self.transform:
            img = self.transform(img)
        return img, torch.tensor(label, dtype=torch.float32)


train_ds = DrowsinessDataset(train_cached, train_transform)
val_ds   = DrowsinessDataset(val_cached, val_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

print(f'Train batches: {len(train_loader)}  |  Val batches: {len(val_loader)}')

imgs, labels = next(iter(train_loader))
print(f'Batch shape: {imgs.shape}  |  Labels: {labels[:8].tolist()}')

## Step 7 — Model: ResNet-50 Backbone + Spatial Attention + Classifier

**Architecture:**
- ResNet-50 (pretrained, outputs 7x7x2048 feature maps)
- **Channel-Spatial Attention** (CBAM-style) — the novelty component
- Global Average Pool → FC → Sigmoid

In [ ]:
class ChannelAttention(nn.Module):
    """Squeeze-and-Excitation style channel attention."""
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        b, c, _, _ = x.size()
        avg_out = self.fc(self.avg_pool(x).view(b, c))
        max_out = self.fc(self.max_pool(x).view(b, c))
        attn = self.sigmoid(avg_out + max_out).view(b, c, 1, 1)
        return x * attn


class SpatialAttention(nn.Module):
    """Spatial attention via channel-wise pooling + conv."""
    def __init__(self, kernel_size=7):
        super().__init__()
        pad = kernel_size // 2
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=pad, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = x.mean(dim=1, keepdim=True)
        max_out = x.max(dim=1, keepdim=True)[0]
        combined = torch.cat([avg_out, max_out], dim=1)
        attn = self.sigmoid(self.conv(combined))
        return x * attn


class CBAM(nn.Module):
    """Convolutional Block Attention Module (channel + spatial)."""
    def __init__(self, channels, reduction=16, spatial_kernel=7):
        super().__init__()
        self.channel_attn = ChannelAttention(channels, reduction)
        self.spatial_attn = SpatialAttention(spatial_kernel)

    def forward(self, x):
        x = self.channel_attn(x)
        x = self.spatial_attn(x)
        return x


class DrowsinessClassifier(nn.Module):
    def __init__(self, backbone_name='resnet50', pretrained=True):
        super().__init__()
        backbone = getattr(models, backbone_name)(
            weights='IMAGENET1K_V2' if pretrained else None
        )
        self.features = nn.Sequential(*list(backbone.children())[:-2])  # up to last conv block
        feat_channels = 2048

        self.attention = CBAM(feat_channels, reduction=16, spatial_kernel=7)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(feat_channels, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(512, 1),
        )

    def forward(self, x):
        feats = self.features(x)         # [B, 2048, 7, 7]
        feats = self.attention(feats)     # attention-refined features
        pooled = self.pool(feats).flatten(1)  # [B, 2048]
        return self.classifier(pooled).squeeze(1)  # [B]


model = DrowsinessClassifier().to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable   = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params:     {total_params:,}')
print(f'Trainable params: {trainable:,}')

## Step 8 — Loss, Optimizer & Scheduler

In [ ]:
n_drowsy     = sum(1 for _, l in train_cached if l == 1)
n_non_drowsy = sum(1 for _, l in train_cached if l == 0)
pos_weight = torch.tensor([n_non_drowsy / max(n_drowsy, 1)], device=DEVICE)
print(f'Class balance — Drowsy: {n_drowsy}, Non Drowsy: {n_non_drowsy}')
print(f'pos_weight (to handle imbalance): {pos_weight.item():.3f}')

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)

## Step 9 — Training Loop with Early Stopping
*[RUN ONCE — results saved to Drive]*

In [ ]:
HISTORY_CACHE = os.path.join(RESULTS_DIR, 'step9_history.pkl')
CKPT_PATH = os.path.join(ARTIFACTS_DIR, 'best_model.pt')

if os.path.exists(CKPT_PATH) and os.path.exists(HISTORY_CACHE):
    with open(HISTORY_CACHE, 'rb') as f:
        history = pickle.load(f)
    best_val_auc = max(history['val_auc'])
    model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE, weights_only=True))
    print(f'[CACHED] Training results loaded from: {HISTORY_CACHE}')
    print(f'  Epochs trained: {len(history["train_loss"])}')
    print(f'  Best validation AUC: {best_val_auc:.4f}')
    print(f'  Model checkpoint: {CKPT_PATH}')
else:
    scaler = torch.amp.GradScaler(enabled=(DEVICE.type == 'cuda'))

    history = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'val_auc': []}
    best_val_auc = 0.0
    patience, patience_counter = 5, 0

    for epoch in range(1, NUM_EPOCHS + 1):
        model.train()
        running_loss = 0.0
        for imgs, labels in tqdm(train_loader, desc=f'Epoch {epoch}/{NUM_EPOCHS} [Train]', leave=False):
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            with torch.amp.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == 'cuda')):
                logits = model(imgs)
                loss = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            running_loss += loss.item() * imgs.size(0)
        train_loss = running_loss / len(train_ds)

        model.eval()
        val_loss = 0.0
        all_probs, all_labels = [], []
        with torch.no_grad():
            for imgs, labels in tqdm(val_loader, desc=f'Epoch {epoch}/{NUM_EPOCHS} [Val]', leave=False):
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                logits = model(imgs)
                val_loss += criterion(logits, labels).item() * imgs.size(0)
                probs = torch.sigmoid(logits)
                all_probs.extend(probs.cpu().tolist())
                all_labels.extend(labels.cpu().tolist())

        val_loss /= len(val_ds)
        preds = [1 if p > 0.5 else 0 for p in all_probs]
        val_acc = sum(p == l for p, l in zip(preds, all_labels)) / len(all_labels)
        val_auc = roc_auc_score(all_labels, all_probs)

        scheduler.step()

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['val_auc'].append(val_auc)

        lr_now = optimizer.param_groups[0]['lr']
        print(f'Epoch {epoch:02d}  '
              f'train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  '
              f'val_acc={val_acc:.4f}  val_AUC={val_auc:.4f}  lr={lr_now:.2e}')

        if val_auc > best_val_auc:
            best_val_auc = val_auc
            patience_counter = 0
            torch.save(model.state_dict(), CKPT_PATH)
            print(f'  -> New best model saved (AUC={val_auc:.4f})')
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f'  Early stopping at epoch {epoch}')
                break

    print(f'\nBest validation AUC: {best_val_auc:.4f}')

    with open(HISTORY_CACHE, 'wb') as f:
        pickle.dump(history, f)
    print(f'Saved training history to: {HISTORY_CACHE}')

## Step 10 — Training Curves
*[Loads cached history from Drive]*

In [ ]:
epochs_ran = range(1, len(history['train_loss']) + 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(epochs_ran, history['train_loss'], label='Train')
axes[0].plot(epochs_ran, history['val_loss'], label='Val')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend()

axes[1].plot(epochs_ran, history['val_acc'])
axes[1].set_title('Val Accuracy'); axes[1].set_xlabel('Epoch')
axes[1].set_ylim(0, 1)

axes[2].plot(epochs_ran, history['val_auc'])
axes[2].set_title('Val AUC-ROC'); axes[2].set_xlabel('Epoch')
axes[2].set_ylim(0, 1)

plt.tight_layout()
plt.show()

## Step 11 — Evaluation on Best Model (Confusion Matrix + ROC + Report)
*[RUN ONCE — results saved to Drive]*

In [ ]:
EVAL_CACHE = os.path.join(RESULTS_DIR, 'step11_eval.pkl')

if os.path.exists(EVAL_CACHE):
    with open(EVAL_CACHE, 'rb') as f:
        eval_data = pickle.load(f)
    all_probs = eval_data['all_probs']
    all_labels = eval_data['all_labels']
    print(f'[CACHED] Loaded evaluation results from: {EVAL_CACHE}')
else:
    model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE, weights_only=True))
    model.eval()

    all_probs, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in tqdm(val_loader, desc='Final evaluation'):
            imgs = imgs.to(DEVICE)
            logits = model(imgs)
            all_probs.extend(torch.sigmoid(logits).cpu().tolist())
            all_labels.extend(labels.tolist())

    with open(EVAL_CACHE, 'wb') as f:
        pickle.dump({'all_probs': all_probs, 'all_labels': all_labels}, f)
    print(f'Saved evaluation results to: {EVAL_CACHE}')

preds = [1 if p > 0.5 else 0 for p in all_probs]

print('\n=== Classification Report ===')
print(classification_report(all_labels, preds, target_names=['Non-Drowsy', 'Drowsy']))

cm = confusion_matrix(all_labels, preds)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].imshow(cm, cmap='Blues')
axes[0].set_title('Confusion Matrix')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')
axes[0].set_xticks([0,1]); axes[0].set_yticks([0,1])
axes[0].set_xticklabels(['Non-Drowsy','Drowsy'])
axes[0].set_yticklabels(['Non-Drowsy','Drowsy'])
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, str(cm[i][j]), ha='center', va='center',
                     color='white' if cm[i][j] > cm.max()/2 else 'black', fontsize=16)

fpr, tpr, _ = roc_curve(all_labels, all_probs)
auc_val = roc_auc_score(all_labels, all_probs)
axes[1].plot(fpr, tpr, label=f'AUC = {auc_val:.4f}')
axes[1].plot([0,1],[0,1], 'k--', alpha=0.4)
axes[1].set_title('ROC Curve'); axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
axes[1].legend()

plt.tight_layout()
plt.show()

## Step 12 — Visualize Attention Maps
*[RUN ONCE — results saved to Drive]*

Show what the attention layer focuses on for a few sample images.

In [ ]:
ATTN_CACHE = os.path.join(RESULTS_DIR, 'step12_attention.pkl')

inv_normalize = T.Normalize(
    mean=[-0.485/0.229, -0.456/0.224, -0.406/0.225],
    std=[1/0.229, 1/0.224, 1/0.225],
)

if os.path.exists(ATTN_CACHE):
    with open(ATTN_CACHE, 'rb') as f:
        attn_data = pickle.load(f)
    print(f'[CACHED] Loaded attention data from: {ATTN_CACHE}')
else:
    def get_attention_map(model, img_tensor):
        model.eval()
        activation = {}

        def hook_fn(module, inp, out):
            activation['spatial_attn'] = out

        handle = model.attention.spatial_attn.conv.register_forward_hook(hook_fn)

        with torch.no_grad():
            _ = model(img_tensor.unsqueeze(0).to(DEVICE))

        handle.remove()
        attn_map = torch.sigmoid(activation['spatial_attn']).squeeze().cpu().numpy()
        return attn_map

    sample_indices = random.sample(range(len(val_ds)), 4)
    attn_data = []
    for idx in sample_indices:
        img_tensor, label = val_ds[idx]
        attn = get_attention_map(model, img_tensor)
        img_display = inv_normalize(img_tensor).permute(1, 2, 0).clamp(0, 1).numpy()
        attn_resized = np.array(Image.fromarray(attn).resize((IMG_SIZE, IMG_SIZE)))
        attn_data.append({'img': img_display, 'attn': attn_resized, 'label': int(label)})

    with open(ATTN_CACHE, 'wb') as f:
        pickle.dump(attn_data, f)
    print(f'Saved attention data to: {ATTN_CACHE}')

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i, item in enumerate(attn_data):
    axes[0, i].imshow(item['img'])
    axes[0, i].set_title('Drowsy' if item['label'] == 1 else 'Non Drowsy')
    axes[0, i].axis('off')

    axes[1, i].imshow(item['img'])
    axes[1, i].imshow(item['attn'], cmap='jet', alpha=0.5)
    axes[1, i].set_title('Attention overlay')
    axes[1, i].axis('off')

plt.suptitle('Top: Original  |  Bottom: Spatial Attention Heatmap', fontsize=14)
plt.tight_layout()
plt.show()

## Step 13 — Single-Image Inference Demo

Load any face image and predict drowsy/non-drowsy.

In [ ]:
def predict_single_image(image_path, model, mtcnn_model, device=DEVICE):
    model.eval()
    img = Image.open(image_path).convert('RGB')

    face = mtcnn_model(img)
    if face is not None:
        face_pil = T.ToPILImage()(face.byte())
    else:
        face_pil = T.Compose([T.Resize(IMG_SIZE+32), T.CenterCrop(IMG_SIZE)])(img)

    face_pil = face_pil.resize((IMG_SIZE, IMG_SIZE))
    tensor = val_transform(face_pil).unsqueeze(0).to(device)

    with torch.no_grad():
        logit = model(tensor)
        prob = torch.sigmoid(logit).item()

    label = 'DROWSY' if prob > 0.5 else 'NON-DROWSY'
    confidence = prob if prob > 0.5 else 1 - prob

    plt.figure(figsize=(5, 5))
    plt.imshow(face_pil)
    plt.title(f'{label}  (confidence: {confidence:.1%})', fontsize=14)
    plt.axis('off')
    plt.show()

    return label, confidence


# --- Example usage ---
# Change the path below to any image you want to test:
test_img = val_cached[0][0]  # first image from validation set
predict_single_image(test_img, model, mtcnn)

## Step 14 — Save Model to Drive

In [ ]:
save_dir = '/content/drive/MyDrive/datasets'
os.makedirs(save_dir, exist_ok=True)
drive_ckpt = os.path.join(save_dir, 'drowsiness_best_model.pt')
torch.save(model.state_dict(), drive_ckpt)
print(f'Model saved to: {drive_ckpt}')